# Control inflation / control completion on MATPOWER case39

This notebook accompanies the new dual PIC construction in the manuscript.

The physical stochastic-control geometry is summarized by
\[
Q=\sigma\sigma^\top,\qquad G=BR^{-1}B^\top.
\]
Exact scalar-temperature path-integral solvability requires \(Q=\lambda G\).  When
\(\lambda G\preceq Q\), there are two dual relaxations:

1. **noise deflation:** \(Q_- = \lambda G\), keeping the physical control geometry;
2. **control inflation/completion:** \(G_+ = Q/\lambda\), keeping the physical noise.

The second problem contains the physical controller as a subset and therefore has value
\(W_+\le J^\star\).  Its physical HJB residual is
\[
\mathcal R[W_+] = \frac12\nabla W_+^\top(G_+-G)\nabla W_+\ge0,
\]
so no curvature assumption is required.

In [ ]:
from pathlib import Path
import json, csv, sys
import numpy as np
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks': ROOT=ROOT.parent
RES=ROOT/'results'
ci=json.loads((RES/'control_inflation_h3.json').read_text())
ci

## Power-system specialization

For nodal injection noise \(\Gamma dW\) and physical injection-control Gramian
\(H=A_gR^{-1}A_g^\top\), the completed Gramian is
\[
H_+(\lambda)=\Gamma^2/\lambda,
\qquad H_v=\Gamma^2/\lambda-H\succeq0.
\]
A factor \(L_vL_v^\top=H_v\) acts as a set of fictitious nodal power controls.

For homogeneous noise, H3, and equal generator price \(r\),
\(\lambda_{\max}=r\sigma^2\) and
\[
H_v=\frac1r(I-S_gS_g^\top).
\]
Thus the **minimal** completion adds only load-bus virtual power control.  This is an oracle/relaxation, not a proposed physical actuator architecture.

In [ ]:
import sys
sys.path.insert(0,str(ROOT/'src'))
from pic_case39 import build_case39_problem
from stochastic_swing import noise_profile
from actuation_hierarchy import build_hierarchy
from control_inflation import control_inflation_geometry
problem,meta=build_case39_problem()
H3=build_hierarchy(problem,meta['bus_ids'])[-1]
for kind in ['homogeneous','activity_scaled']:
    Gamma=noise_profile(problem,0.1,kind)
    g=control_inflation_geometry(problem,H3,Gamma,control_weight=0.2)
    print(kind, 'lambda_max=',g.lam, 'virtual rank=',g.virtual_rank, 'inflation trace=',g.inflation_trace)

## First H3 benchmark

The exact completed value is the Feynman--Kac log-partition function under the **physical** all-bus noise.  The cached numbers below were obtained from three independent importance-sampled replications, using a full-nodal LQR drift only as a Girsanov guide.

Because `-lambda*log(sample mean)` is nonlinear, these raw Monte Carlo estimates should not themselves be described as rigorous numerical lower confidence bounds.  The theorem concerns the exact completed value; a statistically rigorous finite-sample certificate requires a separate confidence construction.

In [ ]:
rows=list(csv.DictReader(open(RES/'h0_h3.csv')))
h3={r['noise']:r for r in rows if r['H']=='H3'}
for display,key in [('homogeneous','homogeneous'),('activity scaled','activity_scaled')]:
    d=h3[display]
    wplus=ci[key]['W_inflation_mean']
    print(display)
    print('  deployed H3 J        =',float(d['J']))
    print('  noise-deflation W-   =',float(d['W']))
    print('  control-inflation W+ =',wplus)
    print('  J-W+                 =',float(d['J'])-wplus)
    print('  replicate sd         =',ci[key]['W_inflation_rep_sd'])

### Interpretation

For homogeneous noise, control completion is both **automatic** as a subsolution and much tighter in this first experiment: the H3 gap falls from roughly \(3.5\times10^{-2}\) for the deflated candidate to about \(6.3\times10^{-3}\) for the completed oracle.

For activity-scaled noise, the ordering reverses slightly.  This is expected: the two relaxations have no universal ordering.  Here the H3 matching temperature is much smaller, so exact completion requires substantially more virtual control authority.  The importance-sampling problem is also harder.

The next production experiment should extend the completed bound across H0--H3 and distinguish:

- the **architecture-specific minimal completion** at each \(\lambda_{\max}(H_k)\), and
- a **common H3-completed oracle**, which is below the entire nested hierarchy and is therefore directly comparable across ranks.

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(ROOT/'figures'/'case39_control_inflation.png')))